# PneumoType upgrades: conformal prediction + per-hospital tuning
This notebook **reuses the CNNs already trained from scratch** in the main PneumoType notebook (loaded from its saved output files), rebuilds the **exact same patient-level splits** (same code, same seed), and adds two upgrades that need no retraining:
1. **Conformal prediction:** confidence with a *mathematical coverage guarantee*.
2. **Per-hospital threshold tuning** for the federated model.

Inputs: the PneumoType notebook output, Chest X-Ray Images (Pneumonia), COVID-19 Radiography Database. *Prototype, not medical advice.*

In [ ]:
import os, re, glob, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
IMG_SIZE = 224
BATCH = 32
EPOCHS = 25         # small CNN: ~30-60 s per epoch on a T4
LR = 1e-3
MODE = "3class"     # "3class" (normal/bacterial/viral) or "binary" (normal/pneumonia)
DATA_ROOT = "/kaggle/input"
CLASSES = ["normal", "bacterial", "viral"] if MODE == "3class" else ["normal", "pneumonia"]
K = len(CLASSES)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
paths = [p for p in glob.glob(f"{DATA_ROOT}/**/*.jpeg", recursive=True) if "__MACOSX" not in p]

def label_of(p):
    folder = os.path.basename(os.path.dirname(p)).upper()
    name = os.path.basename(p).lower()
    if folder == "NORMAL":
        return "normal"
    if folder == "PNEUMONIA":
        if MODE == "binary": return "pneumonia"
        if "bacteria" in name: return "bacterial"
        if "virus" in name: return "viral"
    return None

def patient_of(p):
    name = os.path.basename(p)
    m = re.match(r"(person\d+)_", name, re.I)
    if m: return m.group(1).lower()
    m = re.match(r"(.*?IM-\d+)-\d+", name, re.I)
    if m: return m.group(1).upper()
    return os.path.splitext(name)[0]

df = pd.DataFrame({"path": paths})
df["label"] = df.path.map(label_of)
df = df.dropna(subset=["label"])
df["fname"] = df.path.map(os.path.basename)
df = df.drop_duplicates(subset=["label", "fname"])   # the Kaggle copy has a duplicated nested folder
df["patient"] = df.path.map(patient_of)
df["y"] = df.label.map({c: i for i, c in enumerate(CLASSES)})

print(len(df), "images,", df.patient.nunique(), "patients")
print(df.label.value_counts())
assert len(df) > 1000, "Images not found: check the dataset is attached and DATA_ROOT is right"

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def group_split(d, n_splits, seed):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    a, b = next(sgkf.split(d, d.y, groups=d.patient))
    return d.iloc[a].reset_index(drop=True), d.iloc[b].reset_index(drop=True)

trainval, test_df = group_split(df, 7, SEED)          # ~15% test
train_df, val_df  = group_split(trainval, 6, SEED)    # ~15% val

assert not set(train_df.patient) & set(test_df.patient)
assert not set(val_df.patient) & set(test_df.patient)
for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(n, len(d), d.label.value_counts().to_dict())

In [ ]:
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

def report(name, y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    rec = recall_score(y_true, y_pred, labels=list(range(K)), average=None, zero_division=0)
    yt_p = (y_true > 0).astype(int); yp_p = (y_pred > 0).astype(int)
    row = {"model": name, "accuracy": accuracy_score(y_true, y_pred), "macro_recall": rec.mean()}
    for c, r in zip(CLASSES, rec): row[f"recall_{c}"] = r
    row["pneumonia_recall(any type)"] = recall_score(yt_p, yp_p, zero_division=0)
    row["false_alarms(normal called pneumonia)"] = int(((yt_p == 0) & (yp_p == 1)).sum())
    row["missed_pneumonia(called normal)"] = int(((yt_p == 1) & (yp_p == 0)).sum())
    row[f"DANGER {CLASSES[1]} missed"] = int(((y_true == 1) & (y_pred != 1)).sum())
    return row

results = []
majority = train_df.y.mode()[0]
results.append(report(f"Baseline: always '{CLASSES[majority]}'", test_df.y, [majority] * len(test_df)))
pd.DataFrame(results)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])

def to_model_input(x):
    # x: [B,1,H,W] in [0,1]
    return (x - 0.5) / 0.25                # simple centering; 1 grayscale channel

class XRays(Dataset):
    def __init__(self, d, tf): self.d, self.tf = d.reset_index(drop=True), tf
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        img = Image.open(r.path).convert("L")
        return self.tf(img), int(r.y)

nw = 2
train_dl = DataLoader(XRays(train_df, train_tf), batch_size=BATCH, shuffle=True, num_workers=nw)
val_dl   = DataLoader(XRays(val_df, eval_tf), batch_size=BATCH, num_workers=nw)
test_dl  = DataLoader(XRays(test_df, eval_tf), batch_size=BATCH, num_workers=nw)

In [ ]:
def block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.MaxPool2d(2))

class OurCNN(nn.Module):
    def __init__(self):
        super().__init__()
        chans = [1, 32, 64, 128, 256, 512]
        self.features = nn.Sequential(*[block(a, b) for a, b in zip(chans[:-1], chans[1:])])

class PneumoType(nn.Module):
    def __init__(self, n_out=None):
        super().__init__()
        self.backbone = OurCNN()
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(512, n_out or K)
    def forward(self, x):
        f = self.backbone.features(to_model_input(x))
        f = F.adaptive_avg_pool2d(F.relu(f), 1).flatten(1)
        return self.head(self.drop(f))
    def embed(self, x):
        f = self.backbone.features(to_model_input(x))
        return F.adaptive_avg_pool2d(F.relu(f), 1).flatten(1)

model = PneumoType().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")

In [ ]:
counts = train_df.y.value_counts().sort_index().values
class_w = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float32, device=device)

@torch.no_grad()
def predict_probs(dl, m=None):
    m = m or model; m.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            P.append(F.softmax(m(x.to(device)).float(), 1).cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

def train_model(m, dl, path, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss(weight=class_w)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
    best = -1
    for ep in range(epochs):
        m.train(); t0 = time.time(); tot = 0
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                loss = criterion(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y)
        sched.step()
        pv, yv = predict_probs(val_dl, m)
        mr = recall_score(yv, pv.argmax(1), average="macro")
        print(f"epoch {ep+1}: train loss {tot/len(dl.dataset):.3f} | val macro recall {mr:.3f} | {time.time()-t0:.0f}s")
        if mr > best:
            best = mr; torch.save(m.state_dict(), path)
    m.load_state_dict(torch.load(path))
    print("best val macro recall:", round(best, 3))
    return best

WEIGHTS = {os.path.basename(q): q for q in glob.glob("/kaggle/input/**/*.pt", recursive=True)}
print("found weights:", sorted(WEIGHTS))
model.load_state_dict(torch.load(WEIGHTS["best.pt"], map_location=device)); model.eval()

In [ ]:
p_test, y_test = predict_probs(test_dl)
pred_test = p_test.argmax(1)
results.append(report("Our CNN (from scratch)", y_test, pred_test))
display(pd.DataFrame(results).round(3))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test, labels=list(range(K))),
                       display_labels=CLASSES).plot(cmap="Blues")
plt.title("Test set confusion matrix"); plt.show()

In [ ]:
def adjusted_pred(p, bonus):
    q = p.copy(); q[:, 1] = q[:, 1] + bonus
    return q.argmax(1)

p_val, y_val = predict_probs(val_dl)
TARGET = 0.95
chosen = 0.0
for b in np.arange(0, 0.6, 0.01):
    if recall_score(y_val, adjusted_pred(p_val, b), labels=[1], average=None)[0] >= TARGET:
        chosen = b; break
print(f"{CLASSES[1]} bonus chosen on validation:", round(chosen, 2))

results.append(report(f"Our CNN + safety bonus {chosen:.2f}", y_test, adjusted_pred(p_test, chosen)))
display(pd.DataFrame(results).round(3))

# Unsure band
for cut in [0.5, 0.6, 0.7, 0.8]:
    conf = p_test.max(1); keep = conf >= cut
    acc = accuracy_score(y_test[keep], adjusted_pred(p_test, chosen)[keep]) if keep.any() else float("nan")
    print(f"cutoff {cut}: decides on {keep.mean():.0%} of X-rays, accuracy on those {acc:.3f}, refers {(~keep).sum()}")

In [ ]:
def shifted_tf(kind):
    ops = [T.Resize((IMG_SIZE, IMG_SIZE))]
    if kind == "low contrast": ops.append(T.ColorJitter(contrast=(0.5, 0.5)))
    if kind == "blur": ops.append(T.GaussianBlur(5, sigma=1.5))
    if kind == "low resolution": ops = [T.Resize((96, 96)), T.Resize((IMG_SIZE, IMG_SIZE))]
    ops.append(T.ToTensor()); return T.Compose(ops)


def loader(d, tf): return DataLoader(XRays(d, tf), batch_size=BATCH, num_workers=nw)

In [ ]:
model2 = PneumoType().to(device)
model2.load_state_dict(torch.load(WEIGHTS['best_robust.pt'], map_location=device)); model2.eval()
print('loaded CNN 2')

In [ ]:
tta_tf = T.Compose([T.Resize((248, 248)), T.CenterCrop(IMG_SIZE), T.ToTensor()])

def all_probs(d):
    p1, y = predict_probs(loader(d, eval_tf), model); p2, _ = predict_probs(loader(d, eval_tf), model2)
    t1, _ = predict_probs(loader(d, tta_tf), model);  t2, _ = predict_probs(loader(d, tta_tf), model2)
    return {"CNN 1": p1, "CNN 2 (robust)": p2, "Ensemble": (p1 + p2) / 2,
            "Ensemble + TTA": (p1 + p2 + t1 + t2) / 4}, y

val_c, y_val = all_probs(val_df)
test_c, y_test = all_probs(test_df)
sel = pd.DataFrame({"val macro recall": {k: recall_score(y_val, v.argmax(1), average="macro") for k, v in val_c.items()},
                    "test macro recall": {k: recall_score(y_test, v.argmax(1), average="macro") for k, v in test_c.items()}})
display(sel.round(3))
FINAL = sel["val macro recall"].idxmax()
print("Final model chosen on validation:", FINAL)
p_val_f, p_test_f = val_c[FINAL], test_c[FINAL]

In [ ]:
def ece(p, y, bins=10):
    conf, pred = p.max(1), p.argmax(1); e = 0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e

def temp_scale(p, Tt):
    z = np.log(np.clip(p, 1e-8, 1)) / Tt
    z = np.exp(z - z.max(1, keepdims=True)); return z / z.sum(1, keepdims=True)

def nll(p, y): return -np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-8, 1)))

TEMP = min(np.arange(0.3, 4.01, 0.05), key=lambda t: nll(temp_scale(p_val_f, t), y_val))
p_val_cal, p_test_cal = temp_scale(p_val_f, TEMP), temp_scale(p_test_f, TEMP)
print(f"temperature = {TEMP:.2f} | test ECE before {ece(p_test_f, y_test):.3f} -> after {ece(p_test_cal, y_test):.3f}")

fig, ax = plt.subplots(figsize=(5, 5))
for name, p in [("before", p_test_f), ("after", p_test_cal)]:
    conf, cor = p.max(1), (p.argmax(1) == y_test)
    edges = np.linspace(0, 1, 11); xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 5: xs.append(conf[m].mean()); ys.append(cor[m].mean())
    ax.plot(xs, ys, "o-", label=f"{name} calibration")
ax.plot([0, 1], [0, 1], "--", c="gray", label="perfect"); ax.set_xlabel("model confidence"); ax.set_ylabel("actual accuracy")
ax.legend(); ax.set_title("Reliability diagram (test)"); plt.show()

In [ ]:
FINAL_BONUS = 0.0
for b in np.arange(0, 1.0, 0.01):
    if recall_score(y_val, adjusted_pred(p_val_cal, b), labels=[1], average=None)[0] >= TARGET:
        FINAL_BONUS = b; break
print(f"final {CLASSES[1]} bonus (validation): {FINAL_BONUS:.2f}")

pred_final = p_test_cal.argmax(1)
pred_final_safe = adjusted_pred(p_test_cal, FINAL_BONUS)
results.append(report(f"FINAL: {FINAL}", y_test, pred_final))
results.append(report(f"FINAL: {FINAL} + safety bonus {FINAL_BONUS:.2f}", y_test, pred_final_safe))
display(pd.DataFrame(results).round(3))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred_final, labels=list(range(K))), display_labels=CLASSES).plot(cmap="Blues")
plt.title(f"FINAL ({FINAL}) confusion matrix, test"); plt.show()

pat_idx = test_df.groupby("patient").indices; pats = list(pat_idx)
rng = np.random.default_rng(SEED); boot = []
for _ in range(1000):
    idx = np.concatenate([pat_idx[p] for p in rng.choice(pats, len(pats))])
    yt, yp = y_test[idx], pred_final[idx]
    boot.append([accuracy_score(yt, yp), recall_score(yt, yp, average="macro"),
                 recall_score((yt > 0).astype(int), (yp > 0).astype(int), zero_division=0)])
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
point = [accuracy_score(y_test, pred_final), recall_score(y_test, pred_final, average="macro"),
         recall_score((y_test > 0).astype(int), (pred_final > 0).astype(int))]
ci = pd.DataFrame({"value": point, "95% CI low": lo, "95% CI high": hi},
                  index=["accuracy", "macro recall", "pneumonia recall (any type)"])
display(ci.round(3))

In [ ]:
# Unsure band on the final calibrated model
unsure_rows = []
for cut in [0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = p_test_cal.max(1) >= cut
    acc = accuracy_score(y_test[keep], pred_final_safe[keep]) if keep.any() else float("nan")
    unsure_rows.append({"confidence cutoff": cut, "share decided": keep.mean(), "accuracy on decided": acc, "referred to doctor": int((~keep).sum())})
unsure_df = pd.DataFrame(unsure_rows); display(unsure_df.round(3))

In [ ]:
import copy
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

md_files = glob.glob(f"{DATA_ROOT}/**/Normal.metadata.xlsx", recursive=True)
ADULT_OK = len(md_files) > 0
print("Adult dataset found:", ADULT_OK)
if ADULT_OK:
    aroot = os.path.dirname(md_files[0])
    lookup = {os.path.splitext(os.path.basename(q))[0].lower(): q for q in glob.glob(f"{aroot}/**/*.png", recursive=True)
              if "/masks/" not in q.replace("\\\\", "/")}
    def load_cls(cls, label):
        m = pd.read_excel(f"{aroot}/{cls}.metadata.xlsx")
        m.columns = [str(c).strip().upper() for c in m.columns]
        d = pd.DataFrame({"path": m["FILE NAME"].astype(str).str.lower().map(lookup),
                          "url": m["URL"].astype(str), "source": cls, "y": label})
        return d.dropna(subset=["path"])
    a_norm = load_cls("Normal", 0)
    before = len(a_norm)
    a_norm = a_norm[~a_norm.url.str.contains("chest-xray-pneumonia|paultimothymooney", case=False)]
    print(f"removed {before - len(a_norm)} 'normal' images that came from the children's dataset")
    a_lo, a_cov = load_cls("Lung_Opacity", 1), load_cls("COVID", 1)
    adult_df = pd.concat([a_norm.sample(min(3000, len(a_norm)), random_state=SEED),
                          a_lo.sample(min(1500, len(a_lo)), random_state=SEED),
                          a_cov.sample(min(1500, len(a_cov)), random_state=SEED)]).reset_index(drop=True)
    adult_df["patient"] = adult_df.path
    a_tr, a_rest = train_test_split(adult_df, test_size=0.3, stratify=adult_df.y, random_state=SEED)
    a_va, a_te = train_test_split(a_rest, test_size=0.5, stratify=a_rest.y, random_state=SEED)
    print(adult_df.groupby(["source", "y"]).size())
    binarize = lambda d: d.assign(y=(d.y > 0).astype(int)).reset_index(drop=True)
    k_tr, k_va, k_te = binarize(train_df), binarize(val_df), binarize(test_df)
    a_tr, a_va, a_te = [d.reset_index(drop=True) for d in (a_tr, a_va, a_te)]

In [ ]:
if ADULT_OK:
    def dl_(d, tf=eval_tf, shuffle=False): return DataLoader(XRays(d, tf), batch_size=BATCH, shuffle=shuffle, num_workers=nw)
    fed_model = PneumoType(2).to(device)
    fed_model.load_state_dict(torch.load(WEIGHTS["fed_model.pt"], map_location=device)); fed_model.eval()
    print("loaded federated model")

## Upgrade 1: Conformal prediction (guaranteed coverage)
Normal confidence scores can be wrong about themselves. **Split conformal prediction** uses the validation set to find a cutoff so that the tool outputs a **set of possible diagnoses** that contains the true answer at least **95% of the time** (a statistical guarantee, assuming new X-rays come from the same source).
- **One label in the set:** the tool is sure, and it acts on it.
- **Two or three labels** (e.g. {bacterial, viral}): it's genuinely ambiguous, so it **refers to a doctor**. This is a principled version of our "unsure" mode.

In [ ]:
def conformal(p_cal, y_cal, alpha):
    n = len(y_cal); s = 1 - p_cal[np.arange(n), y_cal]
    q = np.quantile(s, min(1, np.ceil((n + 1) * (1 - alpha)) / n), method="higher")
    return q
def pred_sets(p, q): return (1 - p) <= q

conf_rows = []
for alpha in [0.20, 0.10, 0.05]:
    q = conformal(p_val_cal, y_val, alpha); S_ = pred_sets(p_test_cal, q)
    cover = S_[np.arange(len(y_test)), y_test]; size = S_.sum(1); single = size == 1
    row = {"target coverage": f"{1-alpha:.0%}", "actual coverage (test)": cover.mean(), "avg set size": size.mean(),
           "decided alone (1 label)": single.mean(), "accuracy when decided alone": (p_test_cal.argmax(1)[single] == y_test[single]).mean()}
    for c in range(K): row[f"coverage {CLASSES[c]}"] = cover[y_test == c].mean()
    conf_rows.append(row)
conf_df = pd.DataFrame(conf_rows); display(conf_df.round(3))
Q95 = conformal(p_val_cal, y_val, 0.05)
print(f"q-hat for 95% coverage = {Q95:.3f}: a label is in the set if its probability >= {1-Q95:.3f}")
ex = pred_sets(p_test_cal, Q95)
sets = pd.Series([" or ".join(np.array(CLASSES)[r]) for r in ex]).value_counts()
display(sets.rename("number of test X-rays").to_frame())

## Upgrade 2: Per-hospital threshold tuning for the federated model
The federated model ranked X-rays well (high AUC) but used one threshold (0.5) for both hospitals, which made it too cautious on children. Each hospital picks **its own** threshold on **its own validation X-rays** (no data shared), maximising balanced accuracy. We then report the untouched test sets.

In [ ]:
if ADULT_OK:
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score
    th_rows = []
    for hname, (va, te) in {"children": (k_va, k_te), "adults": (a_va, a_te)}.items():
        pv, yv = predict_probs(dl_(va), fed_model); pt, yt = predict_probs(dl_(te), fed_model)
        grid = np.linspace(0.02, 0.98, 97)
        t_best = grid[np.argmax([balanced_accuracy_score(yv, (pv[:, 1] >= t).astype(int)) for t in grid])]
        for name, t in [("one shared threshold (0.5)", 0.5), (f"own threshold ({t_best:.2f})", t_best)]:
            pr = (pt[:, 1] >= t).astype(int)
            th_rows.append({"hospital": hname, "threshold": name, "balanced acc": balanced_accuracy_score(yt, pr),
                            "pneumonia caught": recall_score(yt, pr), "normal cleared": recall_score(1 - yt, 1 - pr),
                            "AUC": roc_auc_score(yt, pt[:, 1])})
    th_df = pd.DataFrame(th_rows); display(th_df.round(3))

In [ ]:
print("=== UPGRADE NUMBERS ===")
print(conf_df.round(3).to_string()); print(f"Q95 = {Q95:.4f}")
print(sets.to_string())
if ADULT_OK: print(th_df.round(3).to_string())